# Finish 2, two cards: K=64 pairlatw64lab60fp64, seeds 0, 1

Resumes the two-card K=64 run (beams + channels 8 + --keep-best --pair-lab --pair-latent 8 --act-hidden 64 --msg-dim 24 --scalar-dim 24 --pair-lab-fp64, 60 epochs) from the
last checkpoints in the mounted output of nsanya/so3c-k64-pairlatw64lab60fp64-seeds01, with the same command and
horizon, one seed per T4. Push only after that kernel has stopped. T4 with
torch 2.11.0+cu128 pinned and asserted. Read against w64s24lab60.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    # the local label matters: "torch==2.11.0" is satisfied by any 2.11.0
    # build, so on Kaggle's 2.11.0+cu130 image (2026-11-07) pip did nothing
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==" + WANT,
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, SEEDS, EPOCHS, MAX_SECONDS = 'pairlatw64lab60fp64', [0, 1], 60, 40500
SRC = 'nsanya/so3c-k64-pairlatw64lab60fp64-seeds01'
OUT = "/kaggle/working/results_k64/" + TAG
CKPT = "/kaggle/working/checkpoints/" + TAG
LOGS = "/kaggle/working/logs/" + TAG
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '64', '--eval-chunk-size', '1024', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile', '--keep-best', '--pair-lab', '--pair-latent', '8', '--act-hidden', '64', '--msg-dim', '24', '--scalar-dim', '24', '--pair-lab-fp64']
BASE[BASE.index("DATA")] = DATA
for d in (OUT, LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('9aa7f0ecf699' + chr(10))
print("data:", DATA, "| bundle", '9aa7f0ecf699')

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# The previous kernel's last checkpoints, from its mounted output. Each must
# agree with the result JSON that kernel wrote when it stopped (epoch, batch
# order) and goes where --resume looks; the result JSON does not, so the
# resumed run is not skipped.
import glob, json, pathlib, shutil, torch
TODO, DONE = [], {}
for s in SEEDS:
    pat = "/kaggle/input/**/%s/" + TAG + "/seed%d/*seed%d.%s"
    cks = glob.glob(pat % ("checkpoints", s, s, "pt"), recursive=True)
    js = glob.glob(pat % ("results_k64", s, s, "json"), recursive=True)
    if len(cks) != 1:
        print("seed %d: %d mounted checkpoints, not finished here: %r" % (s, len(cks), cks))
        continue
    st = torch.load(cks[0], map_location="cpu", weights_only=False)
    rec = json.load(open(js[0])) if len(js) == 1 else None
    print("seed %d: checkpoint at epoch %d (history %d), previous kernel's result at epoch %s"
          % (s, st["epoch"], len(st["history"]), rec["epochs_run"] if rec else None))
    assert len(st["history"]) == st["epoch"], (len(st["history"]), st["epoch"])
    if rec is not None:
        assert rec["epochs_run"] == st["epoch"], (rec["epochs_run"], st["epoch"])
        assert rec["batch_order"]["epoch1"] == st["batch_order_epoch1"], s
    if st["epoch"] >= EPOCHS:
        assert rec is not None, "seed %d finished but its result is not mounted" % s
        DONE[s] = rec
        continue
    ck = pathlib.Path(CKPT, "seed%d" % s, pathlib.Path(cks[0]).name)
    ck.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(cks[0], ck)
    TODO.append(s)
    del st
assert TODO or DONE, "nothing mounted: attach the output of " + SRC
print("to finish:", TODO, "| finished in the previous kernel:", sorted(DONE))

In [ ]:
# Smoke on both cards at once, each card with its own seed: one epoch on 20k
# jets per card, fresh models in separate directories. Each must write a
# result.
import glob, json
smoke = []
for i, s in enumerate(TODO[:NGPU]):
    d = "/kaggle/working/smoke/%s/card%d" % (TAG, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", str(s), "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
works = []
for i, ((d, _), rc) in enumerate(zip(smoke, codes)):
    found = glob.glob(d + "/*.json")
    print("card %d: exit %d, result %s" % (i, rc, bool(found)))
    if found:
        r = json.load(open(found[0]))
        print("   seed %d: %.0f s, compile %s" % (r["seed"], r["walltime_sec"], r.get("compile_check")))
        works.append(i)
    else:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert works or not TODO, "the smoke run failed on every card"
PARALLEL = len(works) >= len(TODO)
print("parallel:" if PARALLEL else "only card(s) %s work: seed %s alone" % (works, TODO[:1]),
      "seeds", TODO if PARALLEL else TODO[:1])

In [ ]:
import glob, time
todo = TODO if PARALLEL else TODO[:1]
procs = []
for i, s in enumerate(todo):
    gpu = i if PARALLEL else works[0]
    procs.append((s, launch(["benchmarks.run_top_tagging"] + BASE +
                            ["--epochs", str(EPOCHS), "--seed", str(s),
                             "--results-dir", OUT + "/seed%d" % s,
                             "--ckpt-dir", CKPT + "/seed%d" % s,
                             "--resume", "--max-seconds", str(MAX_SECONDS)],
                            gpu, LOGS + "/seed%d.log" % s)))
    print("seed %d on card %d" % (s, gpu), flush=True)
BAD = []
for s, p in procs:
    rc = p.wait()
    print("=== seed %d: exit %d ===" % (s, rc))
    print(tail(LOGS + "/seed%d.log" % s, 2000))
    if rc != 0 or not glob.glob(OUT + "/seed%d/*.json" % s):
        BAD.append(s)
# a failed seed is reported after the summary below
print("failed seeds:", BAD or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# The finished runs against the same-stack K=64 references (T4, torch 2.11,
# two-card kernels): pairlab60 (60 epochs; the command without the arm's
# own flags), base60, pairlab120 (continued to 120 epochs) and any extra
# reference. Unpaired, by the reading fixed before the push (final weights):
# confirmed if the mean gains >= +0.0003 AUC on RULE_REF and every run lies
# above both RULE_REF runs; not confirmed if it gains <= +0.0000; otherwise
# unresolved.
import json, glob, math, statistics
REFS, RULE, RULE_REF = {'pairlab60': [{'seed': 0, 'auc': 0.9841706950893652, 'rej30': 1346.0933333333332, 'val_loss': 0.16073009371757507, 'gap': 0.005290258920901486, 'order': '4956a97c5cb4356c52e25711c90c47b09d1f3aa680704b0f679fb5adca1bb0fe'}, {'seed': 1, 'auc': 0.9846654727988882, 'rej30': 1246.3827160493827, 'val_loss': 0.15742020308971405, 'gap': 0.004776320185669203, 'order': '99fdd6358841957e8d3d1294d0b55f55527b4b711eccacdafa8445ab04e904ae'}], 'base60': [{'seed': 0, 'auc': 0.9832658718933044, 'rej30': 1134.3483146067415, 'val_loss': 0.16420768201351166, 'gap': 0.003863774649866536, 'order': '4956a97c5cb4356c52e25711c90c47b09d1f3aa680704b0f679fb5adca1bb0fe'}, {'seed': 1, 'auc': 0.9835673502711528, 'rej30': 1153.7942857142857, 'val_loss': 0.1629628688097, 'gap': 0.004186215181866515, 'order': '99fdd6358841957e8d3d1294d0b55f55527b4b711eccacdafa8445ab04e904ae'}], 'pairlab120': [{'seed': 0, 'auc': 0.9843922986120341, 'rej30': 1238.7361963190185, 'val_loss': 0.15967456996440887, 'gap': 0.006830632051143842, 'order': '4956a97c5cb4356c52e25711c90c47b09d1f3aa680704b0f679fb5adca1bb0fe'}, {'seed': 1, 'auc': 0.9847992103121007, 'rej30': 1364.2837837837837, 'val_loss': 0.15719641745090485, 'gap': 0.006695834114716914, 'order': '99fdd6358841957e8d3d1294d0b55f55527b4b711eccacdafa8445ab04e904ae'}], 'w64s24lab60': [{'seed': 0, 'auc': 0.9849212473649043, 'rej30': 1541.328244274809, 'val_loss': 0.15659087896347046, 'gap': 0.005621258196458645, 'order': '4956a97c5cb4356c52e25711c90c47b09d1f3aa680704b0f679fb5adca1bb0fe'}, {'seed': 1, 'auc': 0.9848434101167008, 'rej30': 1382.972602739726, 'val_loss': 0.15662167966365814, 'gap': 0.008876162487942763, 'order': '99fdd6358841957e8d3d1294d0b55f55527b4b711eccacdafa8445ab04e904ae'}]}, (0.0003, 0.0), 'w64s24lab60'
POOLED_SD, POOLED_DF = (0.00028969561359768256, 2)   # within-configuration sd of base60 and pairlab60
mean = statistics.mean
recs = {s: r for s, r in DONE.items()}
for f in sorted(glob.glob(OUT + "/seed*/*.json")):
    r = json.load(open(f)); recs[r["seed"]] = r
done = []
for s in sorted(recs):
    r = recs[s]; t = r["test_metrics"]; env = r["env"]; h = r["history"]; vl = [e["val_loss"] for e in h]
    assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
    ref = {x["seed"]: x for x in REFS["pairlab60"]}.get(s)
    print("seed %d: %d epochs (history %d), %.2f h cumulative, %d params | compile %s | batch order %s pairlab60's"
          % (s, r["epochs_run"], len(h), r["walltime_sec"] / 3600, r["n_params"], r.get("compile_check"),
             "same as" if ref and ref["order"] == r["batch_order"]["epoch1"] else "DIFFERENT from"))
    print("  AUC %.5f  rej@0.3 %.1f  rej@0.5 %.1f  final val loss %.5f (lowest %.5f at epoch %d)  val - train %+.5f"
          % (t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")), vl[-1],
             min(vl), h[vl.index(min(vl))]["epoch"], vl[-1] - h[-1]["train_loss"]))
    if r.get("test_metrics_best_val"):
        tb = r["test_metrics_best_val"]
        print("  best-val weights (epoch %d): AUC %.5f, rej@0.3 %.1f"
              % (r["best_val_epoch"], tb["test_auc"], tb["bg_rej_30"]))
    if r["epochs_run"] < EPOCHS:
        print("  INCOMPLETE (epoch %d of %d): stopped on the session cap again; the next finish kernel resumes it" % (r["epochs_run"], EPOCHS))
        continue
    done.append({"auc": t["test_auc"], "rej30": t["bg_rej_30"], "val_loss": vl[-1],
                 "gap": vl[-1] - h[-1]["train_loss"]})
for name, rows in REFS.items():
    print("%-11s AUC %s | rej@0.3 %s | final val loss %s | val - train %s"
          % (name, " ".join("%.5f" % x["auc"] for x in rows),
             " ".join("%.1f" % x["rej30"] for x in rows),
             " ".join("%.5f" % x["val_loss"] for x in rows),
             " ".join("%+.5f" % x["gap"] for x in rows)))

def p_two_sided(t, df):
    try:
        from scipy import stats
        return 2 * stats.t.sf(abs(t), df)
    except Exception:
        return float("nan")

if done:
    a = [x["auc"] for x in done]
    for name, rows in REFS.items():
        b = [x["auc"] for x in rows]
        line = "%s (%d runs) - %s: dAUC %+.5f, d rej@0.3 %+.1f, d final val loss %+.5f" % (
            TAG, len(done), name, mean(a) - mean(b),
            mean(x["rej30"] for x in done) - mean(x["rej30"] for x in rows),
            mean(x["val_loss"] for x in done) - mean(x["val_loss"] for x in rows))
        t_p = (mean(a) - mean(b)) / (POOLED_SD * math.sqrt(1 / len(a) + 1 / len(b)))
        line += " | against the pooled sd %.5f: t %.2f, df %d, p %.3f" % (POOLED_SD, t_p, POOLED_DF, p_two_sided(t_p, POOLED_DF))
        print(line)
    ref = [x["auc"] for x in REFS[RULE_REF]]
    d = mean(a) - mean(ref)
    if len(done) < len(SEEDS):
        verdict = "no verdict: %d of %d runs complete" % (len(done), len(SEEDS))
    elif d >= RULE[0] and min(a) > max(ref):
        verdict = "CONFIRMED at K=64"
    elif d <= RULE[1]:
        verdict = "NOT CONFIRMED"
    else:
        verdict = "unresolved"
    print("verdict (%s against %s, dAUC %+.5f, runs above both %s runs: %d of %d): %s"
          % (TAG, RULE_REF, d, RULE_REF, sum(x > max(ref) for x in a), len(a), verdict))
assert not BAD, "seeds failed (exit code or no result): %s" % BAD